In [2]:
# SoRL in modded-gpt compatible fashion (for ultra fast pre-training)
# 1. pre-training demands simple model architecture, even .generate function can be wrapped around the trained model afterwards
# 2. no need to include 'kv-cache' for the pre-training experiment here

In [ ]:
from sorl.model import CausalSelfAttention, Block, GPTConfig
import torch 

# mock input 
x = torch.randn(2, 1024, 768)

config = GPTConfig()
attn = CausalSelfAttention(dim=768, n_head=6)
block = Block(config=config)

y, v1 = attn(x)
x, v1 = block(x, v1, x, None)

In [3]:
import torch 
from sorl.gat import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)


token_ids = torch.randint(0, 128 + 8, (2, 4))
idx = token_ids[:, :-1].contiguous()
target = token_ids[:, 1:].contiguous()


# forward pass ()
ppt = model(idx, target, 1024)

In [4]:
from sorl.gat import parallel_denoise
from sorl.gat import generate 


num_iterations = 5 
memory_span = 1024 
temperature = 0.0

parallel_denoise(model, idx, num_iterations=5, memory_span=1024, temperature=0.0)

generate(model, idx, max_new_tokens=5, abstraction_interval=3)


tensor([[ 42,  86, 110, 129,   0,   0, 129,   0],
        [ 46,  21,   4, 129,   0,   0, 129,   0]])

In [5]:
model.vocab_sizes

tensor([128,   8])

In [10]:
import torch 
from sorl.gat_act import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)

from sorl.gat_act import infer_level

idx = torch.randint(0, model.vocab_sizes.sum(), (2, 4)).contiguous()
levels = infer_level(idx, model.vocab_sizes)
abstract_mask = (levels > 0)


# Test 1. when abstract mask has True values
# ----------------------------------------------------------------
assert abstract_mask[:, 0].sum() == 0, "first token should not be denoised"
abstract_repr = torch.zeros(abstract_mask.sum(), gat_config.n_embd, device=idx.device)

# ppt, logits, abstract_repr, act_logits = model(idx, abstract_repr, abstract_mask, 1024)

self = model 

x = self._forward_pass(idx, abstract_repr, abstract_mask, memory_span=1024)

logits = self.lm_head(x)
logits = 30 * torch.tanh(logits / 30)

def _compute_act(logits, idx, abstract_mask, act_threshold: float = 0.9):

    with torch.no_grad():
        predictions = logits.argmax(dim=-1)

        pred_matches_target = (predictions[:, :-1] == idx[:, 1:])
        trajectory_mask_shifted = ~abstract_mask[:, 1:]

        trajectory_correct = (pred_matches_target & trajectory_mask_shifted).float()
        trajectory_total = trajectory_mask_shifted.float()

        # ACT: deterministic, threshold based
        accuracy = trajectory_correct.sum(dim=1) / (trajectory_total.sum(dim=1) + 1e-8)
        has_abstract = (abstract_mask.sum(dim=1) > 0)
        should_stop = (accuracy > act_threshold) | ~has_abstract

    return should_stop

# Issue #1. should_stop=False, but NO abstraction is available, so there is nothing to recurse on
should_stop = _compute_act(logits, idx, abstract_mask)

In [11]:
predictions = logits.argmax(dim=-1)
trajectory_mask = ~abstract_mask
act_threshold = 0.9 

pred_matches_target = (predictions[:, :-1] == idx[:, 1:])
trajectory_mask_shifted = trajectory_mask[:, 1:]

trajectory_correct = (pred_matches_target & trajectory_mask_shifted).float()
trajectory_total = trajectory_mask_shifted.float()

# ACT: deterministic, threshold based
accuracy = trajectory_correct.sum(dim=1) / (trajectory_total.sum(dim=1) + 1e-8)
should_stop = (accuracy > act_threshold)

In [13]:
trajectory_total

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [ ]:
# argmax based ACT
# ----------------------------------------------------------------
# (1). _forward_pass produces representation, convert to logits (for trajectory tokens), convert to argmax, compare with what's already there


In [6]:
from sorl.gat_act import infer_level, get_logits_mask

iteration = 0 
num_iterations = 5 
temperature = 0.0

levels = infer_level(idx, model.vocab_sizes)

predict_mask = torch.roll(abstract_mask, -1, dims=1) # prev token embedding predict next token
predict_mask[:, -1] = False

abstract_levels = levels[abstract_mask]
mask = get_logits_mask(abstract_levels, model.vocab_sizes)

with torch.no_grad():

    # Compute ACT target
    predicted_tokens = torch.argmax(logits[:, :-1], dim=-1)
    target_tokens = idx[:, 1:]
    
    correct = (predicted_tokens == target_tokens).float()
    accuracy = torch.mean(correct * (~abstract_mask[:, 1:]), dim=1)

    act_target = (accuracy <= 0.9).long() * (iteration < num_iterations - 1) # should continue
    
    # Update abstraction 
    predict_mask = torch.roll(abstract_mask, -1, dims=1) # prev token embedding predict next token
    predict_mask[:, -1] = False

    logits = torch.where(mask, logits[predict_mask], torch.tensor(float('-inf'), device=model.device)) # process on logits via masking

    if temperature == 0.0:
        new_tokens = torch.argmax(logits, dim=-1)
    else:
        new_tokens = torch.multinomial(F.softmax(logits / temperature, dim=-1), num_samples=1).squeeze(-1)

    idx[abstract_mask] = new_tokens

In [7]:
mask.shape, logits.shape

(torch.Size([6, 136]), torch.Size([6, 136]))

In [8]:
predict_mask.shape

torch.Size([2, 4])